<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Diagnostics and Testing

A dashboard that will not open gives you little information about what went wrong. Does the computer recognize the Duckiedrone's name? Can packets reach it? Is the dashboard accepting connections?

You have already met tools that answer these questions separately. Here, we will bring them together, first examining a connection to a Duckiedrone and then building a small local connection ourselves. Seeing both ends of a connection will help us interpret what happens when only one end is visible.

Run checks against an authorized __physical__ Duckiedrone from a separate base-station terminal on its intended network. Without one, read the illustrative outputs for the example physical Duckiedrone `amelia` instead of sending Duckiedrone-directed traffic, then complete the loopback activity in your local Linux environment. A virtual Duckiedrone can be inspected in its own shell, but do not assume its `.local` name or dashboard is reachable from the base station.

## Inspect the current machine

Begin in the base-station terminal for a physical Duckiedrone, or in your chosen local Linux terminal for loopback-only practice:

```bash
hostname
ip -brief address
ip route
```

These familiar commands establish the starting point: which computer you are inspecting, its addresses, and the routes available to it.

Identify an active network interface and its address. A Workspace may show a virtual Ethernet interface where a native Ubuntu computer shows Wi-Fi. A destination-specific route lookup below identifies the interface Linux actually selects for a physical Duckiedrone.

## Discover compatible local devices

If you do not know the Duckiedrone's name, this base-station command can help:

```bash
dts fleet discover
```

Where discovery is available, its refreshing table can show Duckiedrone names and states such as `Booting` or `Ready`. Press `Ctrl-C` when finished. Run `dts` only from the separate base-station terminal, not the Workspace or a Duckiedrone shell.

Discovery depends on announcements reaching your terminal's environment. A missing entry does not prove the Duckiedrone is offline. If you already know its name or current address, skip discovery.

## Resolve names and inspect routes

To inspect an authorized physical Duckiedrone, enter its name in the base-station terminal when prompted:

```bash
read -r -p 'Authorized Duckiedrone name: ' DUCKIEDRONE_NAME
```

Keep using this terminal for the Duckiedrone-directed checks: a variable defined in one shell will not automatically appear in another.

Ask for the Duckiedrone's IPv4 address:

```bash
getent ahostsv4 "$DUCKIEDRONE_NAME.local"
```

Unlike the earlier `getent hosts` command, `ahostsv4` requests IPv4 results. This lets us follow one address family throughout the investigation.

For the example `amelia`, a lookup could return:

```shell
192.168.1.201   STREAM amelia.local
192.168.1.201   DGRAM
192.168.1.201   RAW
```

The address is repeated for different socket types, ways applications use network communication. Here, we need the address in the first column.

Enter __your Duckiedrone's verified IPv4 address__ from the first column of the lookup result or your authorized network's device list. Do not continue with a guessed address:

```bash
read -r -p 'Verified Duckiedrone IPv4 address: ' DUCKIEDRONE_IP
```

If the lookup returns nothing, first check what name you supplied:

```bash
printf '%s\n' "$DUCKIEDRONE_NAME.local"
```

A result of `.local` means the variable is empty. Otherwise, check the spelling and whether the Duckiedrone has finished starting on the intended network. Do not use the example IP address for real tests if you have not verified your Duckiedrone's address.

You can also try `getent hosts "$DUCKIEDRONE_NAME.local"`. If it returns only IPv6, the name resolves, but we have not obtained the IPv4 address used in this exercise.

The router's connected-device or DHCP-lease list can provide another way to find an authorized physical Duckiedrone's address. Identify the Duckiedrone by its hostname or known MAC address. A lease records an assigned address, not proof that the device is currently online.

Once you have a current IPv4 address, ask Linux how it would send packets there:

```bash
ip route get "$DUCKIEDRONE_IP"
```

For illustration, a native Ubuntu base station contacting `amelia` might report:

```shell
192.168.1.201 dev wlan0 src 192.168.1.42
```

Linux would use interface `wlan0` and source address `192.168.1.42`. A result containing `via` identifies a next-hop router. A directly connected destination needs no gateway, so the absence of a default route does not necessarily prevent reaching the Duckiedrone.

This command consults the routing table without sending a packet. It tells us where traffic would go; it does not yet tell us whether replies will return.

## Test reachability with a local listener

From the base station, send three requests to your verified Duckiedrone address:

```bash
ping -c 3 "$DUCKIEDRONE_IP"
```

Using the address avoids another name lookup. A successful summary includes:

```shell
3 packets transmitted, 3 received, 0% packet loss
```

All three ICMP echo requests received replies. This tells us packets and replies traveled between the two endpoints.

If no replies arrive, several explanations remain possible: the Duckiedrone may be unavailable, the path may fail, or a firewall may block ICMP. A failed name lookup is a different result: it prevents a name-based test from finding its destination in the first place.

Even successful `ping` replies do not tell us whether a TCP service is listening. To understand that distinction, create a service you can inspect yourself.

### Try it: build a connection on the same computer

__Netcat__, invoked as `nc`, can listen for a TCP connection or initiate one. Open two terminals in the same local Linux environment; no Duckiedrone or base-station network access is needed.

In terminal 1, start a listener:

```bash
nc -l 127.0.0.1 12345
```

`-l` selects listening mode. The command waits for a connection on port `12345`.

The address `127.0.0.1` is loopback: traffic stays within the current network environment. This lets us study a TCP exchange without involving the Duckiedrone or Wi-Fi.

Before connecting, inspect the listener from terminal 2:

```bash
ss -ltn 'sport = :12345'
```

Look for `LISTEN` and the local address `127.0.0.1:12345`. They show that a TCP listener is waiting at the address and port we selected.

If no row appears, check terminal 1 for an error. If the port is already occupied, choose another high port and use it consistently throughout this activity.

Now connect from terminal 2:

```bash
nc 127.0.0.1 12345
```

Type a short message and press `Enter`. It should appear in terminal 1. Try sending a reply from terminal 1.

You have now observed both ends of a TCP connection: a listener accepted a connection, and data traveled between the terminals. Press `Ctrl-C` in both terminals when finished.

<details>
<summary>Check your result</summary>

Before the connection, `ss` should show a loopback listener on the chosen port. Each terminal should display the text sent from the other. This confirms a local TCP exchange, not that another computer can reach the listener.

</details>

## Test one known authorized service

Return to the same base-station terminal for these checks. Skip this section if you are following the illustrative example without an authorized physical Duckiedrone.

We can now ask a more specific question about the Duckiedrone: is its dashboard port accepting TCP connections?

For an HTTP dashboard using port `80`:

```bash
nc -vz -w 3 "$DUCKIEDRONE_IP" 80
```

Use the externally accessible port from your dashboard URL if it differs.

Here, `-v` prints the result, `-z` checks the connection without sending application data, and `-w 3` limits the connection wait to three seconds.

For `amelia`, an illustrative success message is:

```shell
Connection to 192.168.1.201 80 port [tcp/http] succeeded!
```

Something accepted the TCP connection. That is the same connection-establishment step we observed locally, but we have not yet asked the service for a web page. [Table 1](#table-1) distinguishes the possible results of this TCP check.

<table id="table-1" class="lx-table">
  <caption>Table 1: Evidence from a TCP connection check.</caption>
  <thead>
    <tr><th>Result</th><th>What it tells us</th></tr>
  </thead>
  <tbody>
    <tr><td><code>succeeded</code></td><td>The TCP connection completed.</td></tr>
    <tr><td><code>Connection refused</code></td><td>The connection received an explicit rejection, often because nothing is listening; a firewall can also reject it.</td></tr>
    <tr><td>A timeout</td><td>No connection completed within the wait; the result alone does not identify why.</td></tr>
  </tbody>
</table>

You can try this even if `ping` received no replies: a network may permit TCP while blocking ICMP.

### Ask the web service to respond

Use `curl` to request HTTP response headers:

```bash
curl --ipv4 --verbose --head --max-time 5 "http://$DUCKIEDRONE_NAME.local"
```

Substitute your dashboard's URL if it differs. Keep certificate verification enabled for HTTPS.

`--head` requests headers without downloading the page body. `--verbose` shows connection details, `--ipv4` keeps this investigation on IPv4, and `--max-time 5` limits the complete operation.

For `amelia`, output could include:

```shell
* Connected to amelia.local (192.168.1.201) port 80
HTTP/1.1 200 OK
```

Compare the connection address with `DUCKIEDRONE_IP` so that you are combining observations about the same destination.

`200 OK` means the header request succeeded. [Table 2](#table-2) suggests what to investigate for other status codes.

<table id="table-2" class="lx-table">
  <caption>Table 2: What dashboard HTTP status codes suggest.</caption>
  <thead>
    <tr><th>Status</th><th>What to investigate</th></tr>
  </thead>
  <tbody>
    <tr><td><code>301</code> or <code>302</code></td><td>The server is directing the client to another URL.</td></tr>
    <tr><td><code>401</code> or <code>403</code></td><td>Authentication or access restrictions apply.</td></tr>
    <tr><td><code>404</code></td><td>Check the requested URL and path: the resource was not found.</td></tr>
    <tr><td><code>405</code></td><td>The server may reject the <code>HEAD</code> method; try the intended page in the browser.</td></tr>
  </tbody>
</table>

An HTTP error response is still useful evidence: a web server answered. It helps us focus on the application rather than immediately returning to network setup.

Finally, try the dashboard in the browser, or open it from the base station with:

```bash
dts duckiebot dashboard "$DUCKIEDRONE_NAME"
```

A successful header request does not test the page's scripts or complete operation. If the browser and terminal behave differently, record both results and where each ran.

## Follow the diagnosis chain

We have moved from locating a Duckiedrone to asking its application to respond. [Figure 1](#figure-1) summarizes the stages of evidence for the next time a connection fails.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    correct terminal and network
      |
      v
    name resolves to an address
      |
      v
    route selected
      |
      v
    connection to the intended service
      |
      v
    application response
  </pre>
  <figcaption>
    Figure 1: Evidence sequence for an authorized dashboard check.
  </figcaption>
</figure>

Use this sequence as a reference: continue from what you already know, rather than repeating every check mechanically.

1. __Establish the starting point.__ Use `hostname`, `ip -brief address`, and `ip route` to understand the machine and network from which you are testing.

2. __Check that the Duckiedrone is running.__ Confirm its startup state and intended network connection.

3. __Identify the Duckiedrone if necessary.__ Use discovery where available, or the known name from its setup.

4. __Resolve the name.__ Use `getent` to obtain an address. If lookup fails, investigate the name and lookup process separately from the service.

5. __Inspect the selected route.__ Use `ip route get` with the current address. A route describes where packets would go, not whether they arrive.

6. __Look for replies.__ Use a short `ping` test as supporting evidence. Missing replies do not rule out a working TCP service.

7. __Test the intended TCP port.__ Use `nc` to distinguish a completed connection, an explicit refusal, and a timeout.

8. __Ask the application to respond.__ For the dashboard, use `curl` and interpret the HTTP response before trying the full page.

9. __Inspect the listener when needed.__ If the connection was refused and you can open a Duckiedrone shell, use `ss` where the service runs.

For that last step, a service listening internally on port `80` can be inspected from an authorized Duckiedrone shell with:

```bash
ss -ltn 'sport = :80'
```

Apply the binding concepts from the earlier notebook: a listener on `127.0.0.1:80` does not directly accept connections addressed to the Duckiedrone's network interface. With forwarding, its internal port may differ from the externally accessible port tested with `nc`.

If you cannot open a Duckiedrone shell, the refusal remains useful evidence to report. If you do open one, use `exit` to return when finished.

When asking for help, describe the relevant commands, results, and terminal context. Redact credentials and device-specific identifiers from shared output. "The name resolves, but TCP port 80 is refused" provides a better starting point than "the Duckiedrone is offline."

## Further reading

The manuals for [`getent`](https://man7.org/linux/man-pages/man1/getent.1.html), [`ip route`](https://man7.org/linux/man-pages/man8/ip-route.8.html), [`nc`](https://man.openbsd.org/nc), [`ss`](https://man7.org/linux/man-pages/man8/ss.8.html), and [curl](https://curl.se/docs/manpage.html) describe additional options.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
